# Clase 013 — Type hints y mypy

**Parte 0** · Ramalho cap. 8 + PEP 484.

> 🎯 Tipos como documentación verificable. mypy detecta bugs antes de runtime.

> ⏱️ ~75 min

## ⚙️ Setup

In [ ]:
from typing import Optional, Literal, TypedDict, Protocol, TypeAlias
from dataclasses import dataclass

### 🧭 Intuición previa: un contrato que revisa una herramienta

Pensá los *type hints* como un **contrato que el editor y el verificador revisan por vos**: no cambian *cómo* corre el programa — Python los ignora en runtime —, pero permiten que herramientas como **mypy** y tu IDE detecten errores **antes de ejecutar**, igual que un corrector ortográfico marca una palabra mal escrita mientras escribís. Anotar es escribir ese contrato; correr mypy es pedirle a la herramienta que lo revise.

## 1️⃣ Sintaxis básica

```python
def saludar(nombre: str, formal: bool = False) -> str:
    return f'Buenos días, {nombre}' if formal else f'Hola {nombre}'
```

**Importante**: los tipos son **anotaciones** — Python NO los verifica en runtime. Son para tooling (IDE, mypy, IA).

In [ ]:
def saludar(nombre: str, formal: bool = False) -> str:
    return f'Buenos días, {nombre}' if formal else f'Hola {nombre}'

print(saludar('Ana'))
print(saludar('Bob', formal=True))

# Esto NO falla en runtime (Python no verifica), pero mypy lo detectaría:
print(saludar(123))  # type hint dice str, le pasamos int

## 2️⃣ Tipos compuestos modernos

Desde Python 3.9+, usa **lowercase** built-ins:

```python
# ✅ moderno (3.9+)
def f(xs: list[int], lookup: dict[str, float]) -> tuple[int, str]:
    ...

# ❌ viejo (pre-3.9)
from typing import List, Dict, Tuple
def f(xs: List[int], lookup: Dict[str, float]) -> Tuple[int, str]:
    ...
```

Desde 3.10+, usa `|` para uniones:

```python
# ✅ moderno (3.10+)
def parse(x: str | int) -> float | None:
    ...

# ❌ viejo
from typing import Union, Optional
def parse(x: Union[str, int]) -> Optional[float]:
    ...
```

In [ ]:
# Demo: tipos compuestos
def promedios_por_grupo(
    datos: list[dict[str, float]],
    grupo_key: str = 'grupo',
    valor_key: str = 'valor',
) -> dict[str, float]:
    sumas: dict[str, float] = {}
    conteos: dict[str, int] = {}
    for d in datos:
        g = d[grupo_key]
        sumas[g] = sumas.get(g, 0.0) + d[valor_key]
        conteos[g] = conteos.get(g, 0) + 1
    return {g: sumas[g] / conteos[g] for g in sumas}

datos = [
    {'grupo': 'A', 'valor': 10.0},
    {'grupo': 'A', 'valor': 20.0},
    {'grupo': 'B', 'valor': 5.0},
]
print(promedios_por_grupo(datos))

## 3️⃣ `Optional` vs default

Distinción importante:

```python
def f(x: int = 0):           # x es int; default 0 si no se pasa
def f(x: int | None = None): # x puede ser None — el caller debe decidir
```

El segundo caso obliga al cuerpo a manejar `None`:

In [ ]:
def buscar(nombre: str, default: int | None = None) -> int:
    db = {'Ana': 30, 'Bob': 25}
    if nombre in db:
        return db[nombre]
    if default is None:
        raise KeyError(nombre)
    return default

print(buscar('Ana'))
print(buscar('Cris', default=0))
try:
    buscar('Cris')
except KeyError as e:
    print(f'KeyError: {e}')

## 4️⃣ `TypedDict` — diccionarios con esquema

Útil cuando recibes JSON o configs:

In [ ]:
class PersonaDict(TypedDict):
    nombre: str
    edad: int
    activo: bool

def saludar_persona(p: PersonaDict) -> str:
    return f'{p["nombre"]} ({p["edad"]}) está {"activo" if p["activo"] else "inactivo"}'

p: PersonaDict = {'nombre': 'Ana', 'edad': 30, 'activo': True}
print(saludar_persona(p))

## 5️⃣ `Literal` — valores concretos como tipo

Útil para parámetros que solo aceptan ciertos strings:

In [ ]:
def ordenar(items: list[int], orden: Literal['asc', 'desc'] = 'asc') -> list[int]:
    return sorted(items, reverse=(orden == 'desc'))

print(ordenar([3, 1, 4, 1, 5]))
print(ordenar([3, 1, 4, 1, 5], orden='desc'))
# mypy detectaría: ordenar([1,2], orden='upward')  # 'upward' no es 'asc'|'desc'

## 6️⃣ `Protocol` — duck typing tipado

"Cualquier cosa que tenga estos métodos":

In [ ]:
class TienePromedio(Protocol):
    def promedio(self) -> float: ...

@dataclass
class Curso:
    notas: list[float]
    def promedio(self) -> float:
        return sum(self.notas) / len(self.notas)

@dataclass
class Atleta:
    tiempos: list[float]
    def promedio(self) -> float:
        return sum(self.tiempos) / len(self.tiempos)

def reportar(items: list[TienePromedio]) -> None:
    for it in items:
        print(f'{type(it).__name__}: {it.promedio():.2f}')

reportar([Curso([6.5, 7.0]), Atleta([10.1, 9.8])])

## 7️⃣ Correr mypy

```bash
pip install mypy
mypy archivo.py            # modo permisivo
mypy --strict archivo.py   # modo estricto (recomendado para libs)
```

Config en `pyproject.toml`:

```toml
[tool.mypy]
python_version = "3.12"
strict = true
ignore_missing_imports = true   # libs sin stubs
```

**`# type: ignore`** al final de una línea silencia mypy en esa línea — escape hatch para casos legítimos (libs sin stubs, hacks intencionales).

## 8️⃣ ¿Cuándo SÍ, cuándo NO?

**Sí**:
- APIs públicas (funciones que importan otros)
- Data classes / records
- Lógica de dominio compleja
- Tipos que mejoran autocompletado

**Quizá no**:
- Notebooks puramente exploratorios
- Scripts one-shot
- Cuando el tipo es obvio y agregar ruido (`x = 5  # int` no aporta)

## ✅ Checklist

- [ ] Anoto funciones públicas con tipos en params y retorno
- [ ] Uso `list[int]` (3.9+) en vez de `List[int]`
- [ ] Uso `X | None` (3.10+) en vez de `Optional[X]`
- [ ] Sé correr mypy y leer sus errores
- [ ] Conozco `TypedDict`, `Literal`, `Protocol`

## 📝 Homework

Ver `README.md`. Módulo `analytics.py` con 5+ funciones anotadas, `pyproject.toml` con `[tool.mypy] strict=true`, log de mypy sin errores.

## 📖 Definiciones y características

**Type hint (annotation)**

Anotación de tipo en signature o variable: `def f(x: int) -> str`. **Python NO verifica en runtime** — son metadata leída por IDE/linter/mypy. Disponibles en `f.__annotations__`.

**`Optional[X]` / `X | None`**

Indica que el valor puede ser `X` o `None`. `Optional[X]` ≡ `Union[X, None]` ≡ `X | None` (PEP 604, 3.10+). Usa la sintaxis con `|` en código nuevo.

**`TypedDict`**

Esquema para diccionarios: `class PersonaDict(TypedDict): nombre: str; edad: int`. Permite tipear dicts que vienen de JSON/API sin convertirlos a dataclass.

**`Literal`**

Restringe a un conjunto de valores: `Literal['asc', 'desc']` solo acepta esas 2 strings. Útil para flags, modos, enums simples.

**`Protocol` (structural typing)**

Define interfaz por **estructura** (duck typing tipado): `class TienePromedio(Protocol): def promedio(self) -> float: ...`. Cualquier clase con `.promedio()` la satisface, sin heredarla.

**mypy**

Static type checker oficial. Lee tu código, sigue las anotaciones, reporta inconsistencias antes de ejecutar. Modo `--strict` lo hace exigente; recomendado para librerías.

## ⚠️ Errores comunes

| Síntoma / mensaje | Causa y cómo arreglar |
|---|---|
| `from typing import List` da deprecation warning en mypy | Python 3.9+ usa lowercase: `list[int]` en vez de `List[int]`. **Fix**: actualiza imports + sintaxis. |
| `Optional[int] = 0` confunde | `Optional[int]` admite `None`. Si tu default es `0`, no necesitas Optional. **Fix**: `x: int = 0` (no admite None) vs `x: int | None = None` (admite None). |
| mypy se queja "Cannot find module 'libreria'" | Lib sin type stubs publicados. **Fix**: `pip install types-libreria` si existe (PEP 561), o `[tool.mypy] ignore_missing_imports = true` en pyproject. |
| Anoté pero mypy no encuentra errores | mypy no se llamó. **Fix**: `mypy archivo.py`. No es automático — debe estar en CI o pre-commit. |
| `reveal_type(x)` no existe en runtime | Es exclusivo de mypy: lo lees como output del check, NO ejecutas. Si lo dejas en código que corre, lanza NameError. |

## ❓ Preguntas frecuentes

**❓ ¿Tipo hints son obligatorios?**

**No** — Python no los verifica. Pero son fuertemente recomendados en: APIs públicas (funciones que importan otros), librerías, código compartido. En notebooks exploratorios, casi nunca aportan.

**❓ ¿Slow my code los type hints?**

**No** — son metadata, no impactan runtime. `from __future__ import annotations` además las hace lazy (strings, evaluadas solo si introspeccionas).

**❓ ¿mypy en CI: estricto o permisivo?**

Empieza permisivo (sin `--strict`), arregla lo obvio, luego activa `--strict` gradualmente. Si arrancas estricto en un repo viejo, te ahogas en errores y termina ignorado.

**❓ ¿Y si no sé qué tipo poner?**

`Any` (de `typing`) es válido — desactiva el check para ese caso. Mejor que mentir con un tipo incorrecto. Convención: comentario `# TODO: tipo correcto` para revisión futura.

**❓ ¿Pydantic vs dataclass + type hints?**

**dataclass + hints**: tipos solo a nivel mypy, no en runtime. **Pydantic**: valida en runtime (parsing de JSON con coerción y errores legibles). Para input externo (API, config) → Pydantic. Para internal record → dataclass.

## 🔗 Referencias

- Ramalho, *Fluent Python* 2e, cap. 8
- [typing docs](https://docs.python.org/3/library/typing.html)
- [mypy docs](https://mypy.readthedocs.io/)

➡️ **Siguiente:** [014 — NumPy: tipos, creación, atributos](../014-numpy-tipos-creacion-atributos/README.md)

## ✅ Soluciones de los ejercicios

Intentá resolverlos vos primero; acá está una solución de referencia comentada. Cada celda es autocontenida (re-importa lo que usa) y verifica el resultado con `assert`/`print`.

**Ejercicio 1.** Toma una función sin tipos y anótala completa (parámetros y retorno).

In [ ]:
# Ejercicio 1 - Anotar una funcion completa
import inspect
from typing import get_type_hints

# Version SIN tipos (como saldria de clase 008):
#   def resumen(datos): return {"n": len(datos), "prom": sum(datos)/len(datos)}

# Version ANOTADA: parametros y retorno declarados.
def resumen(datos: list[float]) -> dict[str, float]:
    """Devuelve n y promedio de una lista de numeros."""
    return {"n": float(len(datos)), "prom": sum(datos) / len(datos)}

# Verificamos que las anotaciones existen y son las esperadas.
hints = get_type_hints(resumen)
print("Anotaciones:", hints)
print("Firma:", inspect.signature(resumen))

assert hints["datos"] == list[float]
assert hints["return"] == dict[str, float]
assert resumen([2.0, 4.0, 6.0])["prom"] == 4.0
print("OK: funcion anotada correctamente")


**Ejercicio 2.** Distingue `def f(x: int = 0)` (default 0) de `def f(x: int | None = None)` (puede no haber valor).

In [ ]:
# Ejercicio 2 - Optional vs default
from typing import get_type_hints

def con_default(x: int = 0) -> int:
    # x SIEMPRE es int; si no lo pasan, vale 0.
    return x + 1

def opcional(x: int | None = None) -> int:
    # x puede ser int o None; hay que manejar el None explicitamente.
    if x is None:
        return -1          # caso "no hay valor"
    return x + 1

print("con_default() ->", con_default())        # usa el default 0
print("opcional()    ->", opcional())            # detecta None
print("opcional(10)  ->", opcional(10))

# El tipo declarado lo confirma:
assert get_type_hints(con_default)["x"] == int
assert get_type_hints(opcional)["x"] == (int | None)
assert con_default() == 1 and opcional() == -1 and opcional(10) == 11
print("OK: default=0 NO admite None; int|None=None SI admite None")


**Ejercicio 3.** Define `class PersonaDict(TypedDict)` con `nombre: str`, `edad: int` y úsala como tipo de un parámetro.

In [ ]:
# Ejercicio 3 - TypedDict como esquema de diccionario
from typing import TypedDict, get_type_hints

class PersonaDict(TypedDict):
    nombre: str
    edad: int

def describir(p: PersonaDict) -> str:
    # p es un dict comun en runtime; el tipo es solo para el checker/IDE.
    return f"{p['nombre']} tiene {p['edad']} anios"

ana: PersonaDict = {"nombre": "Ana", "edad": 30}
print(describir(ana))

# El esquema queda accesible por introspeccion:
print("Esquema:", get_type_hints(PersonaDict))
assert get_type_hints(PersonaDict) == {"nombre": str, "edad": int}
assert describir(ana) == "Ana tiene 30 anios"
print("OK: TypedDict define el esquema del dict")


**Ejercicio 4.** Crea un archivo con un bug de tipo intencional (`def f(x: int) -> str: return x + 1`) y corre `mypy`. Si mypy no está instalado, degrada a validar los type hints por introspección.

In [ ]:
# Ejercicio 4 - Correr mypy (con degradacion si no esta instalado)
import importlib.util, subprocess, sys, tempfile, os, textwrap
from typing import get_type_hints

# Codigo con un bug de tipo intencional: promete str pero devuelve int.
codigo_con_bug = textwrap.dedent('''
    def f(x: int) -> str:
        return x + 1   # BUG: x + 1 es int, pero la firma promete str
''')

tiene_mypy = importlib.util.find_spec("mypy") is not None
print("mypy instalado:", tiene_mypy)

if tiene_mypy:
    # Escribimos un archivo temporal y corremos mypy sobre el.
    ruta = os.path.join(tempfile.gettempdir(), "bug_tipos_demo.py")
    with open(ruta, "w", encoding="utf-8") as fh:
        fh.write(codigo_con_bug)
    res = subprocess.run([sys.executable, "-m", "mypy", ruta],
                         capture_output=True, text=True)
    print("--- salida de mypy ---")
    print(res.stdout.strip() or res.stderr.strip())
    # mypy detecta el retorno incompatible: exit code != 0.
    assert res.returncode != 0, "mypy deberia reportar el error de tipo"
    assert "str" in res.stdout.lower()
    print("OK: mypy detecto el retorno incompatible (int donde se prometio str)")
    os.remove(ruta)
else:
    # Degradacion autocontenida: validamos las anotaciones por introspeccion.
    def f(x: int) -> str:
        return x + 1  # el bug sigue ahi, pero sin mypy lo razonamos nosotros
    hints = get_type_hints(f)
    print("Anotaciones de f:", hints)
    # Comprobamos que la funcion PROMETE str aunque en la practica devuelva int:
    assert hints["return"] is str
    assert isinstance(f(1), int)  # evidencia del bug: devuelve int, no str
    print("OK (degradado): la firma promete str pero f(1) devuelve int -> "
          "eso es justo lo que mypy marcaria")


**Ejercicio 5.** Define `class TienePromedio(Protocol)` con método `promedio() -> float` y acepta cualquier clase que lo implemente (duck typing tipado).

In [ ]:
# Ejercicio 5 - Protocol (structural typing)
from typing import Protocol, runtime_checkable

@runtime_checkable
class TienePromedio(Protocol):
    def promedio(self) -> float: ...

class Curso:
    def __init__(self, notas: list[float]) -> None:
        self.notas = notas
    def promedio(self) -> float:          # cumple el Protocol sin heredarlo
        return sum(self.notas) / len(self.notas)

def reportar(obj: TienePromedio) -> str:
    # Acepta CUALQUIER objeto con .promedio() -> float
    return f"promedio = {obj.promedio():.2f}"

c = Curso([4.0, 5.0, 6.0])
print(reportar(c))

# El Protocol se satisface por estructura, no por herencia:
assert isinstance(c, TienePromedio)   # runtime_checkable lo permite verificar
assert reportar(c) == "promedio = 5.00"
print("OK: Curso satisface TienePromedio solo por tener .promedio()")
